# Liver Regression

## Introduction


I am working on a regression problem where my goal is to to investigate whether five liver-related blood-test measurements contain useful information for estimating reported daily alcohol consumption in this historical dataset and how accurately can daily alcohol consumption be estimated from five liver-related blood-test measurements in this dataset and how should the models results be interpreted? And by the end, i should be able to answer this question confidently (:
- Can five liver-related blood-test measurements provide a useful estimate of reported alcohol consumption, and what evidence supports your conclusion?

In [2]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
liver_disorders = fetch_ucirepo(id=60) 
  
# data (as pandas dataframes) 
X = liver_disorders.data.features 
y = liver_disorders.data.targets 
  
# metadata 
print(liver_disorders.metadata) 
  
# variable information 
print(liver_disorders.variables) 



/opt/anaconda3/lib/python3.13/site-packages/seaborn/_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.stats import gaussian_kde


{'uci_id': 60, 'name': 'Liver Disorders', 'repository_url': 'https://archive.ics.uci.edu/dataset/60/liver+disorders', 'data_url': 'https://archive.ics.uci.edu/static/public/60/data.csv', 'abstract': 'BUPA Medical Research Ltd. database donated by Richard S. Forsyth', 'area': 'Health and Medicine', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 345, 'num_features': 5, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['drinks'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2016, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C54G67', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': 'The first 5 variables are all blood tests which are thought to be sensitive to liver disorders that might arise from excessive alcohol consumption. Each line in the dataset constitutes the record of a single male individual.\n\nImportant note: The 7

In [3]:
X.head()

,mcv,alkphos,sgpt,sgot,gammagt
0,85,92,45,27,31
1,85,64,59,32,23
2,86,54,33,16,54
3,91,78,34,24,36
4,87,70,12,28,10


In [9]:
y.head()

,drinks
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


## Understanding the Data

In [10]:
# Checkiing feature data
print("Features shape:", X.shape)
print("\nFeature columns:", X.columns.tolist())

# Checking target data
print("\nTarget shape:", y.shape)
print("\nTarget columns:", y.columns.tolist())

# Display first five rows
display(X.head())

display(y.head())

Features shape: (345, 5)

Feature columns: ['mcv', 'alkphos', 'sgpt', 'sgot', 'gammagt']

Target shape: (345, 1)

Target columns: ['drinks']


,mcv,alkphos,sgpt,sgot,gammagt
0,85,92,45,27,31
1,85,64,59,32,23
2,86,54,33,16,54
3,91,78,34,24,36
4,87,70,12,28,10


,drinks
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


## Combining features and target for exploratory data analysis 

In [11]:
# Combine features and target
df = pd.concat([X, y], axis=1)

# Display first five rows
display(df.head())

# Dataset information
df.info()

# Statistical summary
display(df.describe())

,mcv,alkphos,sgpt,sgot,gammagt,drinks
0,85,92,45,27,31,0.0
1,85,64,59,32,23,0.0
2,86,54,33,16,54,0.0
3,91,78,34,24,36,0.0
4,87,70,12,28,10,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 345 entries, 0 to 344
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   mcv      345 non-null    int64  
 1   alkphos  345 non-null    int64  
 2   sgpt     345 non-null    int64  
 3   sgot     345 non-null    int64  
 4   gammagt  345 non-null    int64  
 5   drinks   345 non-null    float64
dtypes: float64(1), int64(5)
memory usage: 16.3 KB


,mcv,alkphos,sgpt,sgot,gammagt,drinks
count,345.000000,345.000000,345.000000,345.000000,345.000000,345.000000
mean,90.159420,69.869565,30.405797,24.643478,38.284058,3.455072
std,4.448096,18.347670,19.512309,10.064494,39.254616,3.337835
min,65.000000,23.000000,4.000000,5.000000,5.000000,0.000000
25%,87.000000,57.000000,19.000000,19.000000,15.000000,0.500000
50%,90.000000,67.000000,26.000000,23.000000,25.000000,3.000000
75%,93.000000,80.000000,34.000000,27.000000,46.000000,6.000000
max,103.000000,138.000000,155.000000,82.000000,297.000000,20.000000


**What i see in the data**

The dataset has 6 columns and the first 5 are blood test results:
  - mcv: average size of red blood cells
  - alkphos: alkaline phosphotase
  - sgpt: alanine aminotransferase
  - sgot: aspartate aminotransferase
  - gammagt: gamma-glutamyl transpeptidase

The last column drinks is how many alcholoc drinks a person has per day. This is the column i want to predict.
There are 345 rows and that means 345 people have been testet.

## Checking quality of the data

In [12]:
# Check missing values
print(f'Missing values:\n{df.isnull().sum()}')

# Check duplicate records
print(f'\nDuplicate records: {df.duplicated().sum()}')

# Check data types
print(f'\nData types:\n{df.dtypes}')

Missing values:
mcv        0
alkphos    0
sgpt       0
sgot       0
gammagt    0
drinks     0
dtype: int64

Duplicate records: 4

Data types:
mcv          int64
alkphos      int64
sgpt         int64
sgot         int64
gammagt      int64
drinks     float64
dtype: object


**What i see in the data quality check**

I see here that there are no missing values but i see the duplicate records are 4 so that means we need to drop them later. All columns are numbers so five are integers and drinks is a float so that means i dont need to encode anything before training.


## Table of summary of columns
- **X**: Features (independent variables)
- **y**: Target variable (dependent variable)   

| Variable Name | Role    | Type        | Description                                                                   | Units | Missing Values |
|---------------|---------|-------------|-------------------------------------------------------------------------------|-------|----------------|
| mcv           | Feature | Continuous  | mean corpuscular volume                                                       |       | no             |
| alkphos       | Feature | Continuous  | alkaline phosphotase                                                          |       | no             |
| sgpt          | Feature | Continuous  | alanine aminotransferase                                                      |       | no             |
| sgot          | Feature | Continuous  | aspartate aminotransferase                                                    |       | no             |
| gammagt       | Feature | Continuous  | gamma-glutamyl transpeptidase                                                 |       | no             |
| drinks        | Target  | Continuous  | number of half-pint equivalents of alcoholic beverages drunk per day          |       | no             |
| selector      | Other   | Categorical | field created by the BUPA researchers to split the data into train/test sets |       | no             |
